# 02 - Data Quality dan Profiling Lanjut

Notebook ini melanjutkan hasil `01_data_profiling.ipynb`. Tahap awal mencakup pembacaan data, profiling DuckDB `SUMMARIZE`, serta deteksi missing values pada dataset berita `idn-news-az`.

## Persiapan data

In [ ]:
from pathlib import Path

import duckdb
import polars as pl

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)

cwd = Path.cwd().resolve()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Root project tidak ditemukan.")

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "idn-news-az" / "data_files"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_bersih.parquet"
PARQUET_FILES = sorted(RAW_DIR.glob("*.parquet"))
if not PARQUET_FILES:
    raise FileNotFoundError(f"Tidak ada file Parquet di {RAW_DIR}")

RAW_GLOB = str(RAW_DIR / "*.parquet")
print(f"Polars {pl.__version__} | DuckDB {duckdb.__version__}")
print(f"Jumlah file: {len(PARQUET_FILES):,}")

In [ ]:
raw_lf = pl.scan_parquet(RAW_GLOB)
schema = raw_lf.collect_schema()
required_columns = ["date", "link", "title", "text"]
missing_columns = set(required_columns) - set(schema.names())
if missing_columns:
    raise KeyError(f"Kolom wajib tidak ditemukan: {sorted(missing_columns)}")

jumlah_baris = raw_lf.select(pl.len()).collect().item()
print(f"Ukuran awal: {jumlah_baris:,} baris x {len(schema)} kolom")
raw_lf.head(5).collect()

## 1. Profiling dengan DuckDB `SUMMARIZE`

DuckDB membaca kumpulan file Parquet secara langsung untuk merangkum tipe data, nilai minimum/maksimum, jumlah unik, dan persentase null.

In [ ]:
raw_sql_path = Path(RAW_GLOB).as_posix().replace("'", "''")
with duckdb.connect() as connection:
    relation = connection.sql(
        f"SUMMARIZE SELECT * FROM read_parquet('{raw_sql_path}')"
    )
    profil = pl.DataFrame(
        relation.fetchall(),
        schema=[column[0] for column in relation.description],
        orient="row",
    )

profil

## 2. Deteksi Missing Values

Selain null asli, nilai kosong dan literal `NaT` pada kolom tanggal diperlakukan sebagai missing value semantik.

In [ ]:
null_counts = raw_lf.select(
    *[pl.col(column).null_count().alias(column) for column in required_columns]
).collect()

null_summary = (
    null_counts
    .transpose(include_header=True, header_name="kolom", column_names=["jumlah_null"])
    .with_columns(
        (pl.col("jumlah_null") / jumlah_baris * 100).round(2).alias("persen_null")
    )
    .sort("jumlah_null", descending=True)
)
null_summary

In [ ]:
semantic_missing = raw_lf.select(
    pl.col("date").str.strip_chars().eq("NaT").sum().alias("date_NaT"),
    *[
        pl.col(column).str.strip_chars().eq("").sum().alias(f"{column}_kosong")
        for column in required_columns
    ],
).collect()
semantic_missing

### Catatan tahap 1

Dataset tidak memiliki null asli pada empat kolom utama. Masalah missing value terdapat pada kolom `date` dalam bentuk literal `NaT`; strategi penanganannya diterapkan pada tahap berikutnya.